[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saha23s/CD-MSC/blob/aaron%2Fpreprocessing/colab_evaluate.ipynb)

# CD-MSC — Checkpoint Evaluation & Submission

Loads any saved checkpoint from Drive and evaluates it on the **test split**.
Outputs:
- BAseen / BAunseen / DSG and per-species recall table
- `submission.txt` in challenge format (`file_id,predicted_species_id`)
- `predictions_with_probs.jsonl` with full softmax probabilities (needed for ensembling)

**No GPU required** — inference on precomputed features is fast on CPU.

In [ ]:
# Clone repo + install deps
import os
if not os.path.exists('/content/CD-MSC'):
    !git clone https://github.com/saha23s/CD-MSC.git /content/CD-MSC
%cd /content/CD-MSC
!git checkout aaron/preprocessing
!git pull origin aaron/preprocessing
!pip install -q -r requirements.txt

In [ ]:
# Mount Drive + restore precomputed features
from google.colab import drive
drive.mount('/content/drive')

import shutil, pathlib
src = pathlib.Path('/content/drive/MyDrive/CD-MSC-feature')
dst = pathlib.Path('Development_data/feature')
dst.mkdir(parents=True, exist_ok=True)
if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print(f'Features restored: {len(list(dst.glob("*.pkl")))} pkl files')
else:
    print('ERROR: MyDrive/CD-MSC-feature not found — run colab_quickstart.ipynb first')

In [ ]:
# List available checkpoints on Drive
import pathlib
DRIVE_OUTPUTS = pathlib.Path('/content/drive/MyDrive/CD-MSC-outputs')

print('Available experiments on Drive:')
for p in sorted(DRIVE_OUTPUTS.iterdir()):
    if not p.is_dir():
        continue
    has_best  = (p / 'model' / 'model_best.pth').exists()
    has_final = (p / 'model' / 'model_final.pth').exists()
    has_cfg   = (p / 'resolved_config.json').exists()
    flags = ' '.join(filter(None, [
        'best'  if has_best  else '',
        'final' if has_final else '',
        'cfg'   if has_cfg   else '',
    ]))
    print(f'  [{flags:18s}] {p.name}')

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Set EXP_NAME to the experiment folder shown above.
# Exp 4 (C-DANN + balanced) is our best: BAunseen=0.2626

EXP_NAME        = 'MTRCNN_seed42_B64_E100_earlystop_min20_pati10_dann0.3_cdann_balanced'
CHECKPOINT_FILE = 'model_best.pth'   # or 'model_final.pth'
EVAL_SPLIT      = 'test'             # 'test' for dev test set; 'validation' to check val metrics
# ──────────────────────────────────────────────────────────────────────────────

EXP_DIR    = DRIVE_OUTPUTS / EXP_NAME
CHECKPOINT = EXP_DIR / 'model' / CHECKPOINT_FILE
CFG_PATH   = EXP_DIR / 'resolved_config.json'

assert EXP_DIR.exists(),    f'Not found: {EXP_DIR}'
assert CHECKPOINT.exists(), f'Checkpoint missing: {CHECKPOINT}'
assert CFG_PATH.exists(),   f'resolved_config.json missing: {CFG_PATH}'
print(f'Experiment : {EXP_NAME}')
print(f'Checkpoint : {CHECKPOINT_FILE}')
print(f'Split      : {EVAL_SPLIT}')

In [ ]:
# Load config, build model, load checkpoint weights
import json, sys, torch
sys.path.insert(0, '/content/CD-MSC')

from framework.utilization import build_model

with open(CFG_PATH) as f:
    config = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = build_model(config, device)
ckpt  = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f'Loaded: epoch={ckpt.get("epoch", "?")}')
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

In [ ]:
# Run evaluation — reuses evaluate.py infrastructure
from evaluate import evaluate_checkpoint, load_unseen_domain_by_species, append_official_metrics
from framework.metadata import SPECIES_NAMES, DOMAIN_NAMES, SPECIES_ID_TO_NAME
from framework.engine import balanced_accuracy
import torch

result = evaluate_checkpoint(config, CHECKPOINT, EVAL_SPLIT, return_predictions=True)
metrics     = result['metrics']
predictions = result['predictions']  # list of dicts with file_id, predicted_species_index, species_probabilities, ...

print(f'\n{"="*54}')
print(f'  {EXP_NAME}')
print(f'  checkpoint: {CHECKPOINT_FILE}  |  split: {EVAL_SPLIT}')
print(f'{"="*54}')

if EVAL_SPLIT == 'test':
    print(f'  BAunseen = {metrics["BA_unseen"]:.4f}   ← primary metric')
    print(f'  BAseen   = {metrics["BA_seen"]:.4f}')
    print(f'  DSG      = {metrics["DSG"]:.4f}')
print(f'  Overall BA = {metrics["species_balanced_accuracy"]:.4f}')
print(f'  Clips evaluated: {len(predictions)}')

In [ ]:
# Per-species recall breakdown
import torch
from framework.metadata import SPECIES_NAMES
from evaluate import load_unseen_domain_by_species

unseen_by_species = load_unseen_domain_by_species(config)

all_preds  = torch.tensor([r['predicted_species_index'] for r in predictions])
all_labels = torch.tensor([r['true_species_index']      for r in predictions])

print(f'\n{"Species":35s} {"Recall":>8}  Unseen domain')
print('-' * 60)
for s, name in enumerate(SPECIES_NAMES):
    mask = all_labels == s
    if mask.sum() == 0:
        continue
    recall  = (all_preds[mask] == s).float().mean().item()
    unseen  = unseen_by_species.get(name, '')
    is_unseen_split = (EVAL_SPLIT == 'test' and
                       any(r['true_domain_label'] == unseen
                           for r in predictions if r.get('true_species_label') == name))
    tag = f'  ← unseen ({unseen})' if is_unseen_split else ''
    print(f'{name:35s} {recall:8.3f}{tag}')

In [ ]:
# Write submission TXT + probability JSON to Drive
#
# Submission format (challenge spec):
#   file_id,predicted_species_id
#   CDMSC2026_EVAL_000001,1
#
# predicted_species_id is 1-indexed (model output is 0-indexed, so add 1).
# For dev test set the file_ids are S_X_D_X_X — swap these for CDMSC2026_EVAL_X
# when running against the actual evaluation set.

out_dir = DRIVE_OUTPUTS / EXP_NAME
out_dir.mkdir(parents=True, exist_ok=True)

# Submission TXT
txt_path = out_dir / f'submission_{EVAL_SPLIT}_{CHECKPOINT_FILE.replace(".pth","")}.txt'
with open(txt_path, 'w') as f:
    f.write('file_id,predicted_species_id\n')
    for r in predictions:
        species_id = r['predicted_species_index'] + 1  # 0-indexed → 1-indexed
        f.write(f'{r["file_id"]},{species_id}\n')
print(f'Submission TXT  → {txt_path}  ({len(predictions)} rows)')

# Probability JSON (for ensembling — preserves full softmax distribution per clip)
probs_path = out_dir / f'probs_{EVAL_SPLIT}_{CHECKPOINT_FILE.replace(".pth","")}.jsonl'
with open(probs_path, 'w') as f:
    for r in predictions:
        row = {
            'file_id':               r['file_id'],
            'predicted_species_id':  r['predicted_species_index'] + 1,
            'species_probabilities': r['species_probabilities'],  # list of 9 floats
        }
        f.write(json.dumps(row) + '\n')
print(f'Probabilities   → {probs_path}')
print()
print('To verify: first few rows of submission TXT:')
with open(txt_path) as f:
    for i, line in enumerate(f):
        print(f'  {line}', end='')
        if i >= 5:
            print('  ...')
            break